# Логистическая регрессия с кросс-валидацией

Подготовку признаков выполняем внутри `Pipeline`, как в `f1.py`. Для оценки делим объявления на пять стратифицированных фолдов и усредняем метрики. После оценки обучаем итоговую модель на доступных метках и получаем прогнозы `0/1` для строк выбранного CSV. Присваиваем `0` объявлениям из файлов аномалий в каждом проверочном фолде и при сохранении итогового CSV.

## 1. Загрузить данные

По умолчанию используем `features.csv`. Чтобы проверить сокращённый набор, поменяйте `FEATURES_PATH` на `best_features.csv`. Метки и признаки сопоставляем по `listing_id`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [9]:
REPO_DIR = Path("/Users/ynelo/Documents/github/neuroslop_kygarikovs_hakaton")
FEATURES_PATH = REPO_DIR / "data_temp" / "features.csv"
VERIFIED_PATH = REPO_DIR / "data" / "verified.csv"
ANOMALY_FILES = [
    REPO_DIR / "data_temp/listing_ids_of_bad_time.csv",
    REPO_DIR / "data_temp/listing_id_of_high.csv",
    REPO_DIR / "data_temp/listing_id_of_block.csv",
    REPO_DIR / "data_temp/low_listing_ids.csv",
]
OUTPUT_PATH = Path(f"submission_logreg_grouped_{FEATURES_PATH.stem}.csv")

In [10]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")

feature_columns = features.columns.difference(["seller_id", "is_deal"], sort=False)
X = features[feature_columns]
y = verified.loc[verified.index.isin(X.index), "is_deal"]
X_train = X.loc[y.index]

print(f"Объявлений в матрице: {len(X)}")
print(f"Размеченных объявлений с признаками: {len(y)} из {len(verified)}")

Объявлений в матрице: 15051
Размеченных объявлений с признаками: 294 из 300


## 2. Создать модель

В числовом блоке заменяем бесконечности пропусками, заполняем пропуски медианой и стандартизируем значения. В категориальном блоке заполняем пропуски самым частым значением и применяем one-hot. Каждый фолд получает свои параметры подготовки, рассчитанные на его обучающей части.

Логистическая регрессия использует L2-регуляризацию с `C=1`.

In [11]:
numeric_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = X_train.columns.difference(numeric_columns, sort=False).tolist()


def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=2000),
)

## 3. Оценить качество на пяти фолдах

В оценку входят объявления из `verified.csv`, для которых есть признаки или срабатывает правило из `ANOMALY_FILES`. Делим их на пять стратифицированных фолдов. На каждой итерации обучаем модель на строках с признаками из обучающей части.

В проверочной части применяем то же правило, что перед сохранением CSV. Аномалии получают `is_deal=0`, в том числе при отсутствии признаков. Для расчёта ROC-AUC их оценка также равна нулю. Сравниваем итоговые прогнозы с метками `verified.csv` и усредняем метрики по пяти фолдам.

In [12]:
anomaly_ids = pd.Index(
    pd.concat(
        [pd.read_csv(path, usecols=["listing_id"])["listing_id"] for path in ANOMALY_FILES],
        ignore_index=True,
    ).dropna().drop_duplicates(),
    name="listing_id",
)


def apply_anomaly_overrides(values, anomaly_ids):
    values = values.reindex(values.index.union(anomaly_ids, sort=False))
    values.loc[anomaly_ids] = 0
    return values

In [13]:
evaluation_y = verified.loc[verified.index.isin(X.index.union(anomaly_ids)), "is_deal"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []
fold_predictions = []
for fold, (train, test) in enumerate(cv.split(evaluation_y.index, evaluation_y), start=1):
    y_fit = evaluation_y.iloc[train]
    y_fit = y_fit.loc[y_fit.index.isin(X.index)]
    fold_model = clone(model).fit(X.loc[y_fit.index], y_fit)

    y_test = evaluation_y.iloc[test]
    test_with_features = y_test.index.intersection(X.index, sort=False)
    test_anomalies = y_test.index.intersection(anomaly_ids, sort=False)
    p_deal = pd.Series(
        fold_model.predict_proba(X.loc[test_with_features])[:, 1],
        index=test_with_features,
    )
    p_deal = apply_anomaly_overrides(p_deal, test_anomalies).loc[y_test.index]
    labels = (p_deal > 0.5).astype(int)

    fold_metrics.append({
        "Фолд": fold,
        "f1": f1_score(y_test, labels),
        "precision": precision_score(y_test, labels),
        "recall": recall_score(y_test, labels),
        "roc_auc": roc_auc_score(y_test, p_deal),
        "accuracy": accuracy_score(y_test, labels),
    })
    fold_predictions.append(pd.DataFrame({
        "fold": fold,
        "is_deal_true": y_test,
        "is_deal_pred": labels,
        "p_deal": p_deal,
    }))

oof_predictions = pd.concat(fold_predictions).loc[evaluation_y.index]
by_fold = pd.DataFrame(fold_metrics).set_index("Фолд")
summary = pd.DataFrame({
    "Среднее": by_fold.mean(),
    "Стандартное отклонение": by_fold.std(ddof=0),
})
print(f"Проверено объявлений: {len(oof_predictions)} из {len(verified)}")
print(f"Аномалий в проверке: {oof_predictions.index.isin(anomaly_ids).sum()}")
display(by_fold.round(4))
display(summary.round(4))

Проверено объявлений: 300 из 300
Аномалий в проверке: 8


,f1,precision,recall,roc_auc,accuracy
Фолд,,,,,
1,0.7500,0.7200,0.7826,0.8425,0.8000
2,0.7273,0.7619,0.6957,0.8549,0.8000
3,0.7660,0.7500,0.7826,0.8519,0.8167
4,0.4706,0.7273,0.3478,0.7368,0.7000
5,0.9020,0.8519,0.9583,0.9572,0.9167


,Среднее,Стандартное отклонение
f1,0.7232,0.1402
precision,0.7622,0.0473
recall,0.7134,0.2018
roc_auc,0.8487,0.0698
accuracy,0.8067,0.0688


## 4. Обучить итоговую модель и получить прогнозы

Обучаем модель на всех объявлениях с доступными метками. Для каждой строки выбранного CSV получаем прогноз `is_deal` равный `0` или `1`.

In [14]:
model.fit(X_train, y)
predictions = X.index.to_frame(index=False).assign(is_deal=model.predict(X))

## 5. Проставить нули для аномалий и сохранить CSV

По `listing_id` присваиваем `is_deal=0` строкам из файлов аномалий. Если объявления ещё нет в прогнозе, добавляем строку с нулём. Чтобы обработать остальные файлы, перечислите их в `ANOMALY_FILES`. Каждый файл должен содержать столбец `listing_id`.

In [15]:
labels = predictions.set_index("listing_id")["is_deal"]
submission = apply_anomaly_overrides(labels, anomaly_ids).astype(int).reset_index(name="is_deal")
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Метка 0 проставлена для {len(anomaly_ids)} ID из файлов аномалий")
print(f"Добавлено строк: {len(submission) - len(predictions)}")
print(f"Метки сохранены для {len(submission)} объявлений в {OUTPUT_PATH.resolve()}")
submission.head()

Метка 0 проставлена для 474 ID из файлов аномалий
Добавлено строк: 223
Метки сохранены для 15274 объявлений в /Users/ynelo/Documents/github/neuroslop_kygarikovs_hakaton/submission_logreg_grouped_features.csv


,listing_id,is_deal
0,145079,1
1,100057,0
2,102500,0
3,157287,1
4,110242,0
